In [1]:
import qsvm4eo
import pandas as pd 
import scipy as scp



In [2]:
df_train = pd.read_csv("./../data/train_32.csv")
df_test = pd.read_csv("./../data/test_32.csv")
n = 128 # We keep only the first 100 terms 

def reduce_dataset(df, n):
    df["lat_rank"] = (
        scp.stats.rankdata(df["Latitude"], method="dense") - 1
    )
    df["lon_rank"] = (
        scp.stats.rankdata(df["Longitude"], method="dense") - 1
    )
    df.sort_values(by=["lat_rank", "lon_rank"], inplace=True)
    return df.iloc[:n]

#df_train = reduce_dataset(df_train, n)
#df_test = reduce_dataset(df_test, n)


In [3]:
encoding_train = qsvm4eo.GeneticEncoding(df_train)
graphs = encoding_train.encode()

item index :  0
Gen   0 | best fitness -6.29
Gen  30 | best fitness -0.11
Gen  60 | best fitness -0.05
Gen  90 | best fitness -0.02
Gen 120 | best fitness -0.02
Gen 150 | best fitness -0.02
Gen 180 | best fitness -0.02
item index :  1
Gen   0 | best fitness -6.15
Gen  30 | best fitness -0.20
Gen  60 | best fitness -0.08
Gen  90 | best fitness -0.02
Gen 120 | best fitness -0.02
Gen 150 | best fitness -0.02
Gen 180 | best fitness -0.01
item index :  2
Gen   0 | best fitness -6.03
Gen  30 | best fitness -0.19
Gen  60 | best fitness -0.06
Gen  90 | best fitness -0.02
Gen 120 | best fitness -0.02
Gen 150 | best fitness -0.02
Gen 180 | best fitness -0.02
item index :  3
Gen   0 | best fitness -8.47
Gen  30 | best fitness -0.22
Gen  60 | best fitness -0.03
Gen  90 | best fitness -0.03
Gen 120 | best fitness -0.01
Gen 150 | best fitness -0.01
Gen 180 | best fitness -0.01
item index :  4
Gen   0 | best fitness -8.29
Gen  30 | best fitness -0.09
Gen  60 | best fitness -0.03
Gen  90 | best fitnes

KeyboardInterrupt: 

In [10]:
graphs_train = graphs 
encoding_test = qsvm4eo.GeneticEncoding(df_test)
graphs_test = encoding_test.encode()

item index :  0
Gen   0 | best fitness -16.65
Gen  30 | best fitness -2.99
Gen  60 | best fitness -2.69
Gen  90 | best fitness -2.57
Gen 120 | best fitness -2.57
Gen 150 | best fitness -2.56
Gen 180 | best fitness -2.55
item index :  1
Gen   0 | best fitness -7.89
Gen  30 | best fitness -1.89
Gen  60 | best fitness -1.72
Gen  90 | best fitness -1.69
Gen 120 | best fitness -1.69
Gen 150 | best fitness -1.69
Gen 180 | best fitness -1.63
item index :  2
Gen   0 | best fitness -8.40
Gen  30 | best fitness -3.13
Gen  60 | best fitness -2.89
Gen  90 | best fitness -2.82
Gen 120 | best fitness -2.81
Gen 150 | best fitness -2.79
Gen 180 | best fitness -2.79
item index :  3
Gen   0 | best fitness -13.70
Gen  30 | best fitness -6.86
Gen  60 | best fitness -6.38
Gen  90 | best fitness -6.35
Gen 120 | best fitness -6.34
Gen 150 | best fitness -6.34
Gen 180 | best fitness -6.32
item index :  4
Gen   0 | best fitness -9.81
Gen  30 | best fitness -6.16
Gen  60 | best fitness -5.59
Gen  90 | best fitn

In [13]:
import numpy as np
import qsvm4eo
from qlmaas.qpus import AnalogQPU
from qat.core import Batch, Schedule

def compute_distributions(qbit_coords, excitations=True):
    my_qpu = AnalogQPU()
    duration = 0.66
    schedules = [
        Schedule(drive=qsvm4eo.generate_myqlm_hamiltonian(qbits), tmax=duration)
        for qbits in qbit_coords
    ]
    jobs = [schedule.to_job() for schedule in schedules]

    # Run jobs
    async_result = my_qpu.submit(Batch(jobs))
    results = async_result.join()

    # Get state probabilities
    probs = np.array([[r.probability for r in result] for result in results])
    if excitations:
        return qsvm4eo.compute_excitation_count(probs)
    else:
        return probs
    

probs_train = compute_distributions(graphs_train)
probs_test = compute_distributions(graphs_test)



Submitted a new batch: SJob141585
Submitted a new batch: SJob141586


In [23]:
y_train = df_train['Label'].to_numpy()
y_test = df_test['Label'].to_numpy()

print(y_train)
print(y_test)

[1 1 1 1 1 1 1 1 1 1 1 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 1
 1 1 1 1 1 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 1 1 1 1 1 1
 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 1 1 1 1 1 1 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]
[3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 3 2 2 2 2 2 1 1 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 2 2 2 2 1 1 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 1 1 1 1 2 2 2 1 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 1 1 1 1 2 1 1]


In [20]:
from sklearn.svm import SVC
regularization = 1.0 
y_train = df_train['Label'].to_numpy()
y_test = df_test['Label'].to_numpy()

# Fit the SVM and get the score
print("Fitting the SVM")
kernel = qsvm4eo.Kernel()
gram_train = kernel.compute_gram_train(probs_train)  # Compute the kernel
model = SVC(kernel="precomputed", C=regularization)
model.fit(gram_train, y_train)
train_score = model.score(gram_train, y_train)


# Compute the kernel and score
print("Testing the SVM")
gram_test = kernel.compute_gram_test(probs_test, probs_train)
y_test_pred = model.predict(gram_test)
test_score = model.score(gram_test, y_test)

print("Train acc:", train_score)
print("Test acc:", test_score)

Fitting the SVM
Testing the SVM
Train acc: 0.953125
Test acc: 0.3359375


[3 1 3 3 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 1 1 3
 3 1 1 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 3 3 1 3 1 3 3 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 3 1 1 1 3 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
